# Sinus Approximator

## SKaiNET dependecies

In [1]:
USE {
    repositories {
        mavenLocal()
    }
    dependencies {
        implementation("sk.ainet.app:kotlin-notebook:0.3.0")
    }
}

### Jetbrains libs

In [2]:
%use kandy
%use dataframe

### SKaiNET libs

In [3]:
import sk.ainet.context.DirectCpuExecutionContext
import sk.ainet.context.ExecutionContext
import sk.ainet.context.data
import sk.ainet.execute.context.computation
import sk.ainet.lang.nn.definition
import sk.ainet.lang.nn.network
import sk.ainet.lang.model.dnn.mlp.pretrained.SinusApproximatorWandB
import sk.ainet.lang.tensor.dsl.tensor
import sk.ainet.lang.tensor.pprint
import sk.ainet.lang.tensor.relu
import sk.ainet.lang.types.FP32
import sk.ainet.lang.nn.Module as SKModule

# MLP Model

### Init weights and biases

In [4]:
val sinusApproximatorWandB = SinusApproximatorWandB()

### Create model

In [5]:
fun createModel(context: ExecutionContext) = definition<FP32, Float> {
    network(context) {
        input(1, "input")  // Single input for x value

        // First hidden layer: 1 -> 16 neurons
        dense(16, "hidden-1") {
            // Weights: 16x1 matrix - explicitly defined values
            weights {
                fromArray(
                    sinusApproximatorWandB.getLayer1WandB("").weights
                )
            }
            // Bias: 16 values - explicitly defined
            bias {
                fromArray(
                    sinusApproximatorWandB.getLayer1WandB("").bias
                )
            }
            activation = { tensor -> with(tensor) { relu() } }
        }

        // Second hidden layer: 16 -> 16 neurons
        dense(16, "hidden-2") {
            // Weights: 16x16 matrix - explicitly defined values
            weights {
                fromArray(
                    sinusApproximatorWandB.getLayer2WandB("").weights
                )
            }
            // Bias: 16 values - explicitly defined
            bias {
                fromArray(
                    sinusApproximatorWandB.getLayer2WandB("").bias
                )
            }
            activation = { tensor -> with(tensor) { relu() } }
        }

        // Output layer: 16 -> 1 neuron
        dense(1, "output") {
            // Weights: 1x16 matrix - explicitly defined values
            weights {
                fromArray(
                    sinusApproximatorWandB.getLayer3WandB("").weights
                )
            }

            // Bias: single value - explicitly defined
            bias {
                fromArray(
                    sinusApproximatorWandB.getLayer3WandB("").bias
                )
            }
        }
    }
}


In [6]:
import sk.ainet.lang.model.Model

fun sk.ainet.lang.nn.Module<FP32, Float>.calcSine(ctx: ExecutionContext, angle: Float): Float {
    val model_: sk.ainet.lang.nn.Module<FP32, kotlin.Float> = this
    return computation<Float>(ctx) {
        // Create a simple input tensor compatible with the model's expected input size (1)
        model_.forward(
            data<FP32, Float>(ctx) {
                tensor<FP32, Float>() {
                    // Using shape(1, 1) to represent a single scalar input in 2D form
                    shape(1, 1) {
                        fromArray(
                            floatArrayOf(angle)
                        )
                    }
                }
            }, ctx
        ).data[0, 0]
    }
}



# KAN Sinus

In [7]:
import sk.ainet.context.DirectCpuExecutionContext
import sk.ainet.lang.kan.examples.SineKanPretrained

val ctx = DirectCpuExecutionContext()
val kanModel = SineKanPretrained.create(ctx)


## Calculations

Calculate 50 samples with Math.sin and predict with NN to compare

In [8]:
val ctx = DirectCpuExecutionContext()
val sineNn =  createModel(ctx)


In [9]:
val numSamples = 100
val maxInput = PI.toFloat() / 2f  // π/2

println("Neural Network vs Math.sin() Comparison")
println("=".repeat(50))
println("Input\t\tNetwork Output\tKAN predicted\tMath.sin()\tDifference\tKAN Difference")
println("-".repeat(50))


var totalError = 0.0
var kantotalError = 0.0

for (i in 0 until numSamples) {
    // Generate input value from 0 to π/2
    val x = (i.toFloat() / (numSamples - 1)) * maxInput

    // Get network prediction
    val predicted = sineNn.calcSine(ctx, x)

    // Get network prediction
    val kanPredicted = kanModel.calcSine(ctx, x)



    // Calculate true sin value
    val actual = sin(x.toDouble()).toFloat()

    // Calculate difference
    val difference = abs(predicted - actual)
    totalError += difference.toDouble()

    val kandifference = abs(kanPredicted - actual)
    kantotalError += kandifference.toDouble()


    // Print comparison (every 10th sample for readability)
    if (i % 10 == 0) {
        println("%.4f\t\t%.4f\t\t%.4f\t\t%.4f\t\t%.4f\t\t%.4f".format(x, predicted, kanPredicted, actual, difference, kandifference))
    }
}

val meanAbsoluteError = totalError / numSamples
println("-".repeat(50))
println("Mean Absolute Error: %.6f".format(meanAbsoluteError))
println("Network approximation quality: ${if (meanAbsoluteError < 0.1) "Good" else "Needs improvement"}")



Neural Network vs Math.sin() Comparison
Input		Network Output	KAN predicted	Math.sin()	Difference	KAN Difference
--------------------------------------------------
0,0000		0,0026		0,0000		0,0000		0,0026		0,0000
0,1587		0,1568		0,1579		0,1580		0,0012		0,0001
0,3173		0,3114		0,3123		0,3120		0,0006		0,0002
0,4760		0,4566		0,4580		0,4582		0,0016		0,0002
0,6347		0,5946		0,5933		0,5929		0,0017		0,0003
0,7933		0,7112		0,7124		0,7127		0,0015		0,0003
0,9520		0,8151		0,8149		0,8146		0,0006		0,0004
1,1107		0,8963		0,8956		0,8960		0,0003		0,0004
1,2693		0,9540		0,9552		0,9549		0,0009		0,0003
1,4280		0,9863		0,9895		0,9898		0,0035		0,0003
--------------------------------------------------
Mean Absolute Error: 0,001559
Network approximation quality: Good


In [10]:
val x_values = List(100) { index ->
    (index / (100 - 1).toFloat()) * (PI / 2)
}

val y_values = List(100) { index ->
    sin(x_values[index])
}


val y_nn_values = List(100) { index ->
    // Get network prediction
    sineNn.calcSine(ctx, x_values[index].toFloat())
}

val y_kann_values = List(100) { index ->
    // Get network prediction
    kanModel.calcSine(ctx, x_values[index].toFloat())
}

val df = dataFrameOf(
    "x" to x_values + x_values + x_values,
    "y" to y_values + y_nn_values+y_kann_values,
    "mode" to List(100) { "sin" } + List(100) { "nn" }+ List(100) { "kan" }
)

In [11]:
df.plot {
    line {
        x("x")
        y("y")
        color("mode") {
            scale = categorical("sin" to Color.PURPLE, "nn" to Color.ORANGE, "kan" to Color.RED)
        }
        width = 1.5
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="YFKIIQ"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"mode":["sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","sin","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","nn","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan","kan"],
"x":[0.0,0.015866629548807864,0.03173325909761573,0.04759989010934167,0.06346651819523146,0.07933314628112124,0.09519978021868333,0.11106640245290081,0.1269330363904629,0.142799670328025,0.1586662925622425,0.17453292649980456,0.19039956043736667,0.20626618267158414,0.22213280490580162,0.23799945054670835,0.2538660727809258,0.2697326950151433,0.28559934065605,0.3014659628902675,0.317332585124485,0.3331992307653917,0.3490658529996091,0.3649324752338266,0.38079912087473333,0.39666571970226155,0.4125323653431683,0.428399010984075,0.44426560981160323,0.46013225545250996,0.4759989010934167,0.4918654999209449,0.5077321455618516,0.5235987912027583,0.5394653900302866,0.5553320356711933,0.5711986813121,0.5870652801396282,0.602931925780535,0.6187985714214417,0.63466517024897,0.6505318158898766,0.6663984615307834,0.6822650603583116,0.6981317059992183,0.713998351640125,0.7298649504676532,0.74573159610856,0.7615982417494667,0.7774648405769949,0.7933314394045231,0.8091980850454299,0.8250647306863366,0.8409313763272434,0.85679802196815,0.8726646676090568,0.8885312196232065,0.9043978652641131,0.9202645109050199,0.9361311565459266,0.9519978021868334,0.9678644478277402,0.9837309998418898,0.9995976454827965,1.0154642911237033,1.03133093676461,1.0471975824055166,1.0630641344196663,1.0789307800605732,1.0947974257014799,1.1106640713423865,1.1265307169832934,1.1423973626242,1.1582639146383498,1.1741305602792564,1.1899972059201631,1.20586385156107,1.2217304972019767,1.2375971428428834,1.253463694857033,1.26933034049794,1.2851969861388466,1.3010636317797533,1.31693027742066,1.3327969230615668,1.3486634750757165,1.3645301207166232,1.3803967663575298,1.3962634119984365,1.4121300576393434,1.42799670328025,1.4438632552943997,1.4597299009353064,1.4755965465762133,1.49146319221712,1.5073298378580267,1.5231964834989333,1.539063035513083,1.554929681153